In [5]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa
from scipy.signal import find_peaks
from scipy.ndimage import median_filter
import subprocess
from sklearn.decomposition import PCA


# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
SMILE_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
PITCH_CONFIG = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\spec_pitch.conf"
OPENSMILE_OUT = r"E:\Research Project (Prof. Preeti Rao)\Progress_2026\features_pitch_new"

os.makedirs(OPENSMILE_OUT, exist_ok=True)

WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train_spec_combined"
OUT_TEST = "csv_test_spec_combined"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)
SMOOTH_WIN = 7   # frames (~70 ms)

SR = 16000
WIN_DUR = 0.100
HOP_DUR = 0.010
FREQ_MAX = 2000

# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels

def wheeze_priority_label(labels, t_start, t_end):
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0

def enforce_min_wheeze_duration(labels, min_len=10):
    """
    Keep wheeze (1) only if it lasts at least min_len frames.
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i
            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1
    return labels


def fill_short_zero_gaps(labels, max_gap=4):
    """
    Convert short zero gaps between ones to ones.
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1
    return labels

def suppress_segments_without_low_peaks(preds, n_peaks, peak_thresh=3):
    """
    Remove wheeze segments where no frame has n_peaks < peak_thresh.
    """
    preds = preds.copy()
    n = len(preds)
    i = 0

    while i < n:
        if preds[i] == 1:
            start = i
            while i < n and preds[i] == 1:
                i += 1
            end = i

            # Check if ANY frame in the segment has n_peaks < peak_thresh
            if not np.any(n_peaks[start:end] < peak_thresh):
                preds[start:end] = 0
        else:
            i += 1

    return preds

def extract_pitch_opensmile(wav_path):
    out_csv = os.path.join(
        OPENSMILE_OUT,
        Path(wav_path).stem + ".csv"
    )

    if not os.path.exists(out_csv):
        cmd = [
            SMILE_PATH,
            "-C", PITCH_CONFIG,
            "-I", wav_path,
            "-O", out_csv
        ]
        subprocess.run(cmd, capture_output=True)

        # Fix delimiter
        df = pd.read_csv(out_csv, sep=';')
        df.columns = [c.strip() for c in df.columns]

    return pd.read_csv(out_csv)


# =============================================================================
# FFT FEATURE EXTRACTION (ENHANCED)
# =============================================================================

PITCH_FEATURES = [
    "flux",
    "centroid",
    "maxPos",
    "minPos",
    "logEnergy",
    "rms",
    "F0direction",
    "directionScore",
    "F0Cand[0]", "F0Cand[1]",
    "candVoicing[0]", "candVoicing[1]",
    "candScores[0]", "candScores[1]","nCandidates",
    "candScores[2]", "candVoicing[2]","F0Cand[2]"
]

def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    # ---- Load openSMILE pitch features ----
    pitch_df = extract_pitch_opensmile(wav_path)

    # Ensure time column exists
    time_col = [c for c in pitch_df.columns if "time" in c.lower()][0]
    pitch_df = pitch_df.rename(columns={time_col: "time_s"})

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []

    for i, start in enumerate(range(0, len(audio) - win_len + 1, hop_len)):
        end = start + win_len
        frame = audio[start:end] * np.hanning(win_len)

        X = np.fft.rfft(frame)
        freqs = np.fft.rfftfreq(win_len, d=1/SR)
        mag = np.abs(X)
        mag[0] = 0

        valid = freqs <= FREQ_MAX
        freqs, mag = freqs[valid], mag[valid]

        # ---- FFT features ----
        idx = np.argmax(mag)
        amp = mag[idx]
        freq = freqs[idx]

        p = mag / (np.sum(mag) + 1e-12)
        spec_entropy = -np.sum(p * np.log(p + 1e-12))

        centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-12)
        spec_bandwidth = np.sqrt(
            np.sum(((freqs - centroid) ** 2) * mag) / (np.sum(mag) + 1e-12)
        )

        peak_to_mean = amp / (np.mean(mag) + 1e-12)
        peaks, _ = find_peaks(mag, height=0.3 * amp)
        n_peaks = len(peaks)

        # ---- Time ----
        t_start = start / SR
        t_end = end / SR

        # ---- Label ----
        label = wheeze_priority_label(labels, t_start, t_end)

        # ---- Pitch feature alignment ----
        pitch_row = pitch_df.iloc[min(i, len(pitch_df)-1)]

        pitch_values = [
            pitch_row[f] if f in pitch_df.columns else 0.0
            for f in PITCH_FEATURES
        ]

        rows.append([
            Path(wav_path).name,
            t_start,
            amp,
            freq,
            spec_entropy,
            spec_bandwidth,
            peak_to_mean,
            n_peaks,
            *pitch_values,
            label
        ])

    return pd.DataFrame(rows, columns=[
        "file", "time_step_s",
        "amplitude", "frequency",
        "spec_entropy", "spec_bandwidth",
        "peak_to_mean", "n_peaks",
        *PITCH_FEATURES,
        "label"
    ])


# =============================================================================
# PROCESS TRAIN FILES
# =============================================================================
train_frames = []
for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)

# =============================================================================
# PROCESS TEST FILES
# =============================================================================
test_frames = []
for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)

# =============================================================================
# MODEL TRAINING
# =============================================================================
feature_cols = [
    "amplitude", "frequency",
    "spec_entropy", "spec_bandwidth",
    "peak_to_mean",*PITCH_FEATURES, "n_peaks"
]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test = test_df[feature_cols].values
y_test = test_df["label"].values
# Flip labels: Normal=1, Wheeze=0
# y_train_flipped = 1 - y_train
# y_test_flipped = 1 - y_test

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

n_pos = np.sum(y_train == 1)   # Normal frames
n_neg = np.sum(y_train == 0)   # Wheeze frames

# scale_pos_weight = n_neg / n_pos
# print(f"scale_pos_weight (Normal) = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    # scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_s, y_train)

# =============================================================================
# PREDICTION + TEMPORAL SMOOTHING
# =============================================================================
# Probabilities correspond to flipped "Normal"
test_df["wheeze_prob"] = model.predict_proba(X_test_s)[:, 1]

# Convert back to wheeze probability
# test_df["wheeze_prob"] = 1 - normal_prob

# ---- Temporal smoothing per file ----
test_df["wheeze_prob_smooth"] = (
    test_df.groupby("file")["wheeze_prob"]
    .transform(lambda x: median_filter(x, size=SMOOTH_WIN))
)

# ---- Final decision ----
# ---- Initial frame-wise decision ----
test_df["predicted_label_raw"] = (
    test_df["wheeze_prob_smooth"] > 0.47
).astype(int)

# ---- Apply temporal constraints per file ----
final_preds = []

for fname, df_f in test_df.groupby("file"):
    preds = df_f["predicted_label_raw"].values
    # peaks = df_f["n_peaks"].values

    # Fill short zero gaps (<= 40 ms)
    preds = fill_short_zero_gaps(preds, max_gap=5)
    
    # Minimum wheeze duration (>= 100 ms)
    preds = enforce_min_wheeze_duration(preds, min_len=15)

    # NEW BIAS: suppress segments without low-peak frames
    # preds = suppress_segments_without_low_peaks(preds,peaks,peak_thresh=6)


    # Re-enforce duration (robustness)
    # preds = enforce_min_wheeze_duration(preds, min_len=10)

    final_preds.append(pd.Series(preds, index=df_f.index))

test_df["predicted_label"] = pd.concat(final_preds).sort_index()

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                            target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE")

              precision    recall  f1-score   support

      Normal       0.62      0.66      0.64      9298
      Wheeze       0.84      0.82      0.83     20522

    accuracy                           0.77     29820
   macro avg       0.73      0.74      0.73     29820
weighted avg       0.77      0.77      0.77     29820

Confusion Matrix:
 [[ 6117  3181]
 [ 3706 16816]]

DONE


In [6]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa
from scipy.signal import find_peaks
from scipy.ndimage import median_filter
import subprocess
from sklearn.decomposition import PCA

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
SMILE_PATH = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\bin\SMILExtract.exe"
PITCH_CONFIG = r"E:\Research Project (Prof. Preeti Rao)\opensmile-3.0-win-x64\myconfig\spec_pitch.conf"
OPENSMILE_OUT = r"E:\Research Project (Prof. Preeti Rao)\Progress_2026\features_pitch_new"

os.makedirs(OPENSMILE_OUT, exist_ok=True)

WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train_spec_combined"
OUT_TEST = "csv_test_spec_combined"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)
SMOOTH_WIN = 7   # frames (~70 ms)

SR = 16000
WIN_DUR = 0.100
HOP_DUR = 0.010
FREQ_MAX = 2000

# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels

def wheeze_priority_label(labels, t_start, t_end):
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0

def enforce_min_wheeze_duration(labels, min_len=10):
    """
    Keep wheeze (1) only if it lasts at least min_len frames.
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i
            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1
    return labels


def fill_short_zero_gaps(labels, max_gap=4):
    """
    Convert short zero gaps between ones to ones.
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1
    return labels

def suppress_segments_without_low_peaks(preds, n_peaks, peak_thresh=3):
    """
    Remove wheeze segments where no frame has n_peaks < peak_thresh.
    """
    preds = preds.copy()
    n = len(preds)
    i = 0

    while i < n:
        if preds[i] == 1:
            start = i
            while i < n and preds[i] == 1:
                i += 1
            end = i

            # Check if ANY frame in the segment has n_peaks < peak_thresh
            if not np.any(n_peaks[start:end] < peak_thresh):
                preds[start:end] = 0
        else:
            i += 1

    return preds

def extract_pitch_opensmile(wav_path):
    out_csv = os.path.join(
        OPENSMILE_OUT,
        Path(wav_path).stem + ".csv"
    )

    if not os.path.exists(out_csv):
        cmd = [
            SMILE_PATH,
            "-C", PITCH_CONFIG,
            "-I", wav_path,
            "-O", out_csv
        ]
        subprocess.run(cmd, capture_output=True)

        # Fix delimiter
        df = pd.read_csv(out_csv, sep=';')
        df.columns = [c.strip() for c in df.columns]

    return pd.read_csv(out_csv)


# =============================================================================
# FFT FEATURE EXTRACTION (ENHANCED)
# =============================================================================

PITCH_FEATURES = [
    "flux",
    "centroid",
    "maxPos",
    "minPos",
    "logEnergy",
    "rms",
    "F0direction",
    "directionScore",
    "F0Cand[0]", "F0Cand[1]",
    "candVoicing[0]", "candVoicing[1]",
    "candScores[0]", "candScores[1]","nCandidates",
    "candScores[2]", "candVoicing[2]","F0Cand[2]"
]

def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    # ---- Load openSMILE pitch features ----
    pitch_df = extract_pitch_opensmile(wav_path)

    # Ensure time column exists
    time_col = [c for c in pitch_df.columns if "time" in c.lower()][0]
    pitch_df = pitch_df.rename(columns={time_col: "time_s"})

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []

    for i, start in enumerate(range(0, len(audio) - win_len + 1, hop_len)):
        end = start + win_len
        frame = audio[start:end] * np.hanning(win_len)

        X = np.fft.rfft(frame)
        freqs = np.fft.rfftfreq(win_len, d=1/SR)
        mag = np.abs(X)
        mag[0] = 0

        valid = freqs <= FREQ_MAX
        freqs, mag = freqs[valid], mag[valid]

        # ---- FFT features ----
        idx = np.argmax(mag)
        amp = mag[idx]
        freq = freqs[idx]

        p = mag / (np.sum(mag) + 1e-12)
        spec_entropy = -np.sum(p * np.log(p + 1e-12))

        centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-12)
        spec_bandwidth = np.sqrt(
            np.sum(((freqs - centroid) ** 2) * mag) / (np.sum(mag) + 1e-12)
        )

        peak_to_mean = amp / (np.mean(mag) + 1e-12)
        peaks, _ = find_peaks(mag, height=0.3 * amp)
        n_peaks = len(peaks)

        # ---- Time ----
        t_start = start / SR
        t_end = end / SR

        # ---- Label ----
        label = wheeze_priority_label(labels, t_start, t_end)

        # ---- Pitch feature alignment ----
        pitch_row = pitch_df.iloc[min(i, len(pitch_df)-1)]

        pitch_values = [
            pitch_row[f] if f in pitch_df.columns else 0.0
            for f in PITCH_FEATURES
        ]

        rows.append([
            Path(wav_path).name,
            t_start,
            amp,
            freq,
            spec_entropy,
            spec_bandwidth,
            peak_to_mean,
            n_peaks,
            *pitch_values,
            label
        ])

    return pd.DataFrame(rows, columns=[
        "file", "time_step_s",
        "amplitude", "frequency",
        "spec_entropy", "spec_bandwidth",
        "peak_to_mean", "n_peaks",
        *PITCH_FEATURES,
        "label"
    ])


# =============================================================================
# PROCESS TRAIN FILES
# =============================================================================
train_frames = []
for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)

# =============================================================================
# PROCESS TEST FILES
# =============================================================================
test_frames = []
for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)

# =============================================================================
# MODEL TRAINING
# =============================================================================
feature_cols = [
    "amplitude", "frequency",
    "spec_entropy", "spec_bandwidth",
    "peak_to_mean",*PITCH_FEATURES, "n_peaks"
]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test = test_df[feature_cols].values
y_test = test_df["label"].values
# Flip labels: Normal=1, Wheeze=0
# y_train_flipped = 1 - y_train
# y_test_flipped = 1 - y_test

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ---- PCA ----
# Option A: retain 95% variance (recommended)
pca = PCA(n_components=0.95, random_state=42)

# Option B (alternative): fixed components
# pca = PCA(n_components=25, random_state=42)

X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)

print(f"PCA components retained: {pca.n_components_}")

n_pos = np.sum(y_train == 1)   # Normal frames
n_neg = np.sum(y_train == 0)   # Wheeze frames

# scale_pos_weight = n_neg / n_pos
# print(f"scale_pos_weight (Normal) = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    # scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_pca, y_train)

# =============================================================================
# PREDICTION + TEMPORAL SMOOTHING
# =============================================================================
# Probabilities correspond to flipped "Normal"
test_df["wheeze_prob"] = model.predict_proba(X_test_pca)[:, 1]

# Convert back to wheeze probability
# test_df["wheeze_prob"] = 1 - normal_prob

# ---- Temporal smoothing per file ----
test_df["wheeze_prob_smooth"] = (
    test_df.groupby("file")["wheeze_prob"]
    .transform(lambda x: median_filter(x, size=SMOOTH_WIN))
)

# ---- Final decision ----
# ---- Initial frame-wise decision ----
test_df["predicted_label_raw"] = (
    test_df["wheeze_prob_smooth"] > 0.47
).astype(int)

# ---- Apply temporal constraints per file ----
final_preds = []

for fname, df_f in test_df.groupby("file"):
    preds = df_f["predicted_label_raw"].values
    # peaks = df_f["n_peaks"].values

    # Fill short zero gaps (<= 40 ms)
    preds = fill_short_zero_gaps(preds, max_gap=5)
    
    # Minimum wheeze duration (>= 100 ms)
    preds = enforce_min_wheeze_duration(preds, min_len=15)

    # NEW BIAS: suppress segments without low-peak frames
    # preds = suppress_segments_without_low_peaks(preds,peaks,peak_thresh=6)


    # Re-enforce duration (robustness)
    # preds = enforce_min_wheeze_duration(preds, min_len=10)

    final_preds.append(pd.Series(preds, index=df_f.index))

test_df["predicted_label"] = pd.concat(final_preds).sort_index()

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                            target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE")

PCA components retained: 6
              precision    recall  f1-score   support

      Normal       0.62      0.63      0.63      9298
      Wheeze       0.83      0.83      0.83     20522

    accuracy                           0.77     29820
   macro avg       0.73      0.73      0.73     29820
weighted avg       0.77      0.77      0.77     29820

Confusion Matrix:
 [[ 5896  3402]
 [ 3583 16939]]

DONE
